# Fraunhofer CPG Test Dataset: JSONL -> CSV + CWE Groups

Ноутбук читает `datasets/tasks 3.jsonl`, разворачивает JSONL в `pandas.DataFrame`, сохраняет CSV и формирует группы CWE. Первая рабочая группа: `taint_flow_vuln`.

In [2]:
from pathlib import Path
import json

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install pandas before running this notebook: python3 -m pip install pandas") from exc

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

DATASET_PATH = Path("/Users/nident/Desktop/JOB/ScolTech/FraunhoferCPG_test/datasets/tasks 3.jsonl")
OUTPUT_CSV_PATH = DATASET_PATH.with_suffix(".csv")

DATASET_PATH, OUTPUT_CSV_PATH

(PosixPath('/Users/nident/Desktop/JOB/ScolTech/FraunhoferCPG_test/datasets/tasks 3.jsonl'),
 PosixPath('/Users/nident/Desktop/JOB/ScolTech/FraunhoferCPG_test/datasets/tasks 3.csv'))

## Read JSONL

Каждая строка JSONL содержит верхнеуровневые блоки `vuln`, `task` и `selected`.

In [3]:
def read_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as fh:
        for line_no, line in enumerate(fh, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at line {line_no}: {exc}") from exc
    return records


records = read_jsonl(DATASET_PATH)
len(records)

968

In [4]:
records[0].keys(), records[0]["vuln"].keys(), records[0]["task"].keys()

(dict_keys(['vuln', 'task', 'selected']),
 dict_keys(['id', 'project', 'cve_ids', 'details', 'summary', 'cwe_ids', 'vuln_commit', 'safe_commit', 'snippets', 'contexts']),
 dict_keys(['id', 'similarity', 'confidence', 'reasoning', 'project', 'safe_commit', 'vuln_commit', 'last_commit']))

## Transform To DataFrame

`pd.json_normalize` разворачивает вложенные словари в колонки вида `vuln.id`, `task.project`. Списки и сложные объекты оставляем в отдельных колонках, а для CSV сериализуем их обратно в JSON-строки.

In [5]:
df = pd.json_normalize(records, sep=".")

df["vuln.cwe_ids"] = df["vuln.cwe_ids"].apply(lambda value: value if isinstance(value, list) else [])
df["vuln.cwe_ids_text"] = df["vuln.cwe_ids"].apply(lambda values: ",".join(values))
df["vuln.cwe_count"] = df["vuln.cwe_ids"].apply(len)
df["vuln.cve_ids_text"] = df["vuln.cve_ids"].apply(lambda values: ",".join(values) if isinstance(values, list) else "")
df["vuln.snippet_count"] = df["vuln.snippets"].apply(lambda values: len(values) if isinstance(values, list) else 0)
df["vuln.context_count"] = df["vuln.contexts"].apply(lambda values: len(values) if isinstance(values, list) else 0)

df.shape

(968, 24)

In [6]:
df.head(3)

,selected,vuln.id,vuln.project,vuln.cve_ids,vuln.details,vuln.summary,vuln.cwe_ids,vuln.vuln_commit,vuln.safe_commit,vuln.snippets,vuln.contexts,task.id,task.similarity,task.confidence,task.reasoning,task.project,task.safe_commit,task.vuln_commit,task.last_commit,vuln.cwe_ids_text,vuln.cwe_count,vuln.cve_ids_text,vuln.snippet_count,vuln.context_count
0,False,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,[CVE-2025-52888],### Summary\nA critical XML External Entity (XXE) vulnerability exists in the xunit-xml-plugin used by Allure 2. The plugin fails to securely configure the XML parser (`Documen...,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,[CWE-611],eaa87ff7d93e79074f7a1d785740bd3fed2f89fd,cbcb33719851ff70adce85d38e15d20fc58d4eb7,"[{'vuln_snippet': 'public class JunitXmlPlugin { private void parseRootElement(final Path resultsDirectory, final Path parsedFile, final ...","[{'vuln_context': '/** * Plugin that reads data in JUnit.xml format. * * @since 2.0 */ @SuppressWarnings({""PMD.ExcessiveImports"", ""ClassDataAbstractionCoupling"", ""ClassFanO...",GHSA-8c3x-hq82-gjcm,0.91,0.95,Both vulnerabilities involve improper configuration of DocumentBuilderFactory leading to XXE (CWE-611). Identical root cause: DTD and external entity processing not disabled. S...,HL7/fhir-ig-publisher,3560de2f486d688a3ddcf4aa54d8bdacea380c3d,ba0b48d4ddb4dbffb45e0d45c43069e99a038385,ff300219c66c19d76fb5e08c92a4d487b71cf15c,CWE-611,1,CVE-2025-52888,2,2
1,True,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,[CVE-2025-52888],### Summary\nA critical XML External Entity (XXE) vulnerability exists in the xunit-xml-plugin used by Allure 2. The plugin fails to securely configure the XML parser (`Documen...,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,[CWE-611],eaa87ff7d93e79074f7a1d785740bd3fed2f89fd,cbcb33719851ff70adce85d38e15d20fc58d4eb7,"[{'vuln_snippet': 'public class JunitXmlPlugin { private void parseRootElement(final Path resultsDirectory, final Path parsedFile, final ...","[{'vuln_context': '/** * Plugin that reads data in JUnit.xml format. * * @since 2.0 */ @SuppressWarnings({""PMD.ExcessiveImports"", ""ClassDataAbstractionCoupling"", ""ClassFanO...",GHSA-2466-4485-4pxj,0.85,0.95,Both vulnerabilities are XXE issues (CWE-611) caused by improper XML parser configuration allowing external entity resolution. Attack vectors differ slightly: one exploits HTTP...,Robothy/local-s3,d6ed756ceb30c1eb9d4263321ac683d734f8836f,009901882be8b543e85b67c5ec2e9f30d83d62ef,0244e9bbb5b1a9b6f8fa3f9c19fc9a9243f4fc97,CWE-611,1,CVE-2025-52888,2,2
2,True,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,[CVE-2025-52888],### Summary\nA critical XML External Entity (XXE) vulnerability exists in the xunit-xml-plugin used by Allure 2. The plugin fails to securely configure the XML parser (`Documen...,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,[CWE-611],eaa87ff7d93e79074f7a1d785740bd3fed2f89fd,cbcb33719851ff70adce85d38e15d20fc58d4eb7,"[{'vuln_snippet': 'public class JunitXmlPlugin { private void parseRootElement(final Path resultsDirectory, final Path parsedFile, final ...","[{'vuln_context': '/** * Plugin that reads data in JUnit.xml format. * * @since 2.0 */ @SuppressWarnings({""PMD.ExcessiveImports"", ""ClassDataAbstractionCoupling"", ""ClassFanO...",GHSA-47qw-ccjm-9c2c,0.85,0.95,Both vulnerabilities are XXE issues (CWE-611) caused by improper XML parser configuration allowing external entity expansion. Attack vectors differ slightly: one exploits multi...,Robothy/local-s3,d6ed756ceb30c1eb9d4263321ac683d734f8836f,009901882be8b543e85b67c5ec2e9f30d83d62ef,0244e9bbb5b1a9b6f8fa3f9c19fc9a9243f4fc97,CWE-611,1,CVE-2025-52888,2,2


## Save CSV

In [7]:
def to_csv_safe(value):
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value


if hasattr(df, "map"):
    df_csv = df.map(to_csv_safe)
else:
    df_csv = df.applymap(to_csv_safe)
df_csv.to_csv(OUTPUT_CSV_PATH, index=False)

OUTPUT_CSV_PATH

PosixPath('/Users/nident/Desktop/JOB/ScolTech/FraunhoferCPG_test/datasets/tasks 3.csv')

## CWE Groups

Группы заданы как массивы CWE. Сейчас основная рабочая группа - `taint_flow_vuln`; остальные группы оставлены как расширяемая разметка для следующих итераций.

In [8]:
CWE_GROUPS = {
    # Untrusted input reaches a security-sensitive sink: injection, traversal, parser abuse, SSRF, unsafe deserialization, upload, etc.
    "taint_flow_vuln": [
        "CWE-20",   # Improper Input Validation
        "CWE-22",   # Path Traversal
        "CWE-36",   # Absolute Path Traversal
        "CWE-79",   # Cross-Site Scripting
        "CWE-80",   # XSS - Basic
        "CWE-81",   # XSS - Error Message
        "CWE-90",   # LDAP Injection
        "CWE-91",   # XML Injection
        "CWE-94",   # Code Injection
        "CWE-95",   # Eval Injection
        "CWE-150",  # Improper Neutralization of Escape/Meta/Control Sequences
        "CWE-434",  # Unrestricted Upload
        "CWE-502",  # Deserialization of Untrusted Data
        "CWE-611",  # XXE
        "CWE-915",  # Improperly Controlled Modification of Object Attributes
        "CWE-918",  # SSRF
    ],
    "authorization_missing_security_guard": ["CWE-285", "CWE-862", "CWE-863"],
    "state_machine_business_logic": ["CWE-348", "CWE-352"],
    "race_condition_atomicity": [],
    "authentication_session_lifecycle": ["CWE-287", "CWE-289", "CWE-522", "CWE-523"],
    "exceptional_paths_fail_open": ["CWE-754", "CWE-755"],
    "crypto_security_api_misuse": [
        "CWE-295", 
        "CWE-297",
        "CWE-611"
    ],
    "sensitive_data_exposure": ["CWE-200", "CWE-209", "CWE-532", "CWE-922"],
    "memory_resource_lifetime": ["CWE-400", "CWE-770", "CWE-789"],
    "configuration_dependency_security": [],
}

taint_flow_vuln = CWE_GROUPS["taint_flow_vuln"]
taint_flow_vuln

['CWE-20',
 'CWE-22',
 'CWE-36',
 'CWE-79',
 'CWE-80',
 'CWE-81',
 'CWE-90',
 'CWE-91',
 'CWE-94',
 'CWE-95',
 'CWE-150',
 'CWE-434',
 'CWE-502',
 'CWE-611',
 'CWE-915',
 'CWE-918']

In [9]:
def has_any_cwe(cwe_ids: list[str], group: list[str]) -> bool:
    return bool(set(cwe_ids) & set(group))


for group_name, group_cwes in CWE_GROUPS.items():
    df[group_name] = df["vuln.cwe_ids"].apply(lambda cwes, group_cwes=group_cwes: has_any_cwe(cwes, group_cwes))

taint_flow_df = df[df["taint_flow_vuln"]].copy()
taint_flow_records = taint_flow_df.to_dict("records")

taint_flow_df.shape, len(taint_flow_records)

((694, 34), 694)

## Quick Checks

In [10]:
cwe_counts = (
    df.explode("vuln.cwe_ids")
      .groupby("vuln.cwe_ids", dropna=False)
      .size()
      .sort_values(ascending=False)
      .rename("count")
      .reset_index()
)

cwe_counts.head(40)

,vuln.cwe_ids,count
0,CWE-79,232
1,CWE-22,174
2,CWE-611,108
3,CWE-20,106
4,CWE-862,89
5,CWE-502,81
6,CWE-400,61
7,CWE-770,58
8,CWE-94,47
9,CWE-532,39


In [11]:
group_counts = (
    pd.Series({group_name: int(df[group_name].sum()) for group_name in CWE_GROUPS})
      .sort_values(ascending=False)
      .rename("rows")
      .reset_index()
      .rename(columns={"index": "group"})
)

group_counts

,group,rows
0,taint_flow_vuln,694
1,memory_resource_lifetime,119
2,authorization_missing_security_guard,111
3,sensitive_data_exposure,61
4,crypto_security_api_misuse,24
5,authentication_session_lifecycle,23
6,state_machine_business_logic,18
7,exceptional_paths_fail_open,17
8,race_condition_atomicity,0
9,configuration_dependency_security,0


In [12]:
taint_flow_df[[
    "vuln.id",
    "vuln.project",
    "vuln.summary",
    "vuln.cwe_ids_text",
    "task.id",
    "task.project",
    "selected",
]].head(20)

,vuln.id,vuln.project,vuln.summary,vuln.cwe_ids_text,task.id,task.project,selected
0,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,CWE-611,GHSA-8c3x-hq82-gjcm,HL7/fhir-ig-publisher,False
1,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,CWE-611,GHSA-2466-4485-4pxj,Robothy/local-s3,True
2,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,CWE-611,GHSA-47qw-ccjm-9c2c,Robothy/local-s3,True
3,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,CWE-611,GHSA-5gwh-r76w-934h,jenkinsci/qualys-was-plugin,False
4,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,CWE-611,GHSA-68j8-fp38-p48q,gematik/app-referencevalidator,True
5,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,CWE-611,GHSA-6fhj-vr9j-g45r,CycloneDX/cyclonedx-core-java,False
6,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,CWE-611,GHSA-7jc7-g598-2p64,opensagres/xdocreport,True
7,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,CWE-611,GHSA-fvfq-q238-j7j3,wso2/carbon-mediation,False
8,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,CWE-611,GHSA-g6wm-2v64-wq36,Robothy/local-s3,False
27,GHSA-h856-ffvv-xvr4,jenkinsci/remoting,Jenkins Remoting library arbitrary file read vulnerability,"CWE-22,CWE-754",GHSA-478x-m3mx-7j3f,jenkinsci/htmlpublisher-plugin,True


In [13]:
import os
projects = os.listdir("/Users/nident/Desktop/JOB/ScolTech/CPG_analysis/data/sweep")
projects = [project.replace("__", '/') for project in projects]
projects

['FasterXML/jackson-core',
 'jenkinsci/gatling-plugin',
 'jenkinsci/structs-plugin',
 'jenkinsci/delphix-plugin',
 'allure-framework/allure2',
 'jenkinsci/htmlpublisher-plugin',
 'gematik/app-referencevalidator',
 'apache/jspwiki',
 'x-stream/xstream',
 'CycloneDX/cyclonedx-core-java',
 'outputs',
 'gaul/s3proxy']

In [14]:
mask = taint_flow_df["vuln.project"].isin(projects)
matched = taint_flow_df[mask]

matched.to_csv("/Users/nident/Desktop/JOB/ScolTech/FraunhoferCPG_test/datasets/matched_taint_flow_vuln.csv", index=False)

In [19]:
pd.set_option("display.max_rows", 1000)
matched

,selected,vuln.id,vuln.project,vuln.cve_ids,vuln.details,vuln.summary,vuln.cwe_ids,vuln.vuln_commit,vuln.safe_commit,vuln.snippets,vuln.contexts,task.id,task.similarity,task.confidence,task.reasoning,task.project,task.safe_commit,task.vuln_commit,task.last_commit,vuln.cwe_ids_text,vuln.cwe_count,vuln.cve_ids_text,vuln.snippet_count,vuln.context_count,taint_flow_vuln,authorization_missing_security_guard,state_machine_business_logic,race_condition_atomicity,authentication_session_lifecycle,exceptional_paths_fail_open,crypto_security_api_misuse,sensitive_data_exposure,memory_resource_lifetime,configuration_dependency_security
0,False,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,[CVE-2025-52888],### Summary\nA critical XML External Entity (XXE) vulnerability exists in the xunit-xml-plugin used by Allure 2. The plugin fails to securely configure the XML parser (`Documen...,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,[CWE-611],eaa87ff7d93e79074f7a1d785740bd3fed2f89fd,cbcb33719851ff70adce85d38e15d20fc58d4eb7,"[{'vuln_snippet': 'public class JunitXmlPlugin { private void parseRootElement(final Path resultsDirectory, final Path parsedFile, final ...","[{'vuln_context': '/** * Plugin that reads data in JUnit.xml format. * * @since 2.0 */ @SuppressWarnings({""PMD.ExcessiveImports"", ""ClassDataAbstractionCoupling"", ""ClassFanO...",GHSA-8c3x-hq82-gjcm,0.91,0.95,Both vulnerabilities involve improper configuration of DocumentBuilderFactory leading to XXE (CWE-611). Identical root cause: DTD and external entity processing not disabled. S...,HL7/fhir-ig-publisher,3560de2f486d688a3ddcf4aa54d8bdacea380c3d,ba0b48d4ddb4dbffb45e0d45c43069e99a038385,ff300219c66c19d76fb5e08c92a4d487b71cf15c,CWE-611,1,CVE-2025-52888,2,2,True,False,False,False,False,False,False,False,False,False
1,True,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,[CVE-2025-52888],### Summary\nA critical XML External Entity (XXE) vulnerability exists in the xunit-xml-plugin used by Allure 2. The plugin fails to securely configure the XML parser (`Documen...,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,[CWE-611],eaa87ff7d93e79074f7a1d785740bd3fed2f89fd,cbcb33719851ff70adce85d38e15d20fc58d4eb7,"[{'vuln_snippet': 'public class JunitXmlPlugin { private void parseRootElement(final Path resultsDirectory, final Path parsedFile, final ...","[{'vuln_context': '/** * Plugin that reads data in JUnit.xml format. * * @since 2.0 */ @SuppressWarnings({""PMD.ExcessiveImports"", ""ClassDataAbstractionCoupling"", ""ClassFanO...",GHSA-2466-4485-4pxj,0.85,0.95,Both vulnerabilities are XXE issues (CWE-611) caused by improper XML parser configuration allowing external entity resolution. Attack vectors differ slightly: one exploits HTTP...,Robothy/local-s3,d6ed756ceb30c1eb9d4263321ac683d734f8836f,009901882be8b543e85b67c5ec2e9f30d83d62ef,0244e9bbb5b1a9b6f8fa3f9c19fc9a9243f4fc97,CWE-611,1,CVE-2025-52888,2,2,True,False,False,False,False,False,False,False,False,False
2,True,GHSA-h7qf-qmf3-85qg,allure-framework/allure2,[CVE-2025-52888],### Summary\nA critical XML External Entity (XXE) vulnerability exists in the xunit-xml-plugin used by Allure 2. The plugin fails to securely configure the XML parser (`Documen...,Allure Report allows Improper XXE Restriction via DocumentBuilderFactory,[CWE-611],eaa87ff7d93e79074f7a1d785740bd3fed2f89fd,cbcb33719851ff70adce85d38e15d20fc58d4eb7,"[{'vuln_snippet': 'public class JunitXmlPlugin { private void parseRootElement(final Path resultsDirectory, final Path parsedFile, final ...","[{'vuln_context': '/** * Plugin that reads data in JUnit.xml format. * * @since 2.0 */ @SuppressWarnings({""PMD.ExcessiveImports"", ""ClassDataAbstractionCoupling"", ""ClassFanO...",GHSA-47qw-ccjm-9c2c,0.85,0.95,Both vulnerabilities are XXE issues (CWE-611) caused by improper XML parser configuration allowing external entity expansion. Attack vectors differ slightly: one exploits multi...,Robothy/local-s3,d6ed756ceb30c1eb9d4263321ac683d734f8836f

In [18]:
525/5

105.0